# Step 2: Configuration (`core/config.py`)

**Goal:** One place in the codebase that owns API keys, model catalogue, and pricing.
Everything else asks config — nothing talks to the environment directly.

---

## Why centralise config?

If you scatter `os.getenv("OPENAI_API_KEY")` across 5 files:
- When the env var name changes, you update 5 places
- Error messages are inconsistent (or missing)
- There's no single place to see what keys the app needs

One config module = one place to look, one place to change.

---

## Why NOT fetch pricing dynamically?

This is a common beginner question — worth addressing directly.

**None of the three providers offer an official pricing API.**
Your only options would be:
1. **Scrape their pricing pages** — brittle, breaks when their HTML changes
2. **Use a third-party pricing service** — may itself be stale, adds a dependency

Even [LiteLLM](https://github.com/BerriAI/litellm) — a widely used production library — hardcodes pricing
and updates it via version releases.

**The pragmatic approach (what we do):**
- Hardcode rates with a `# Last updated: YYYY-MM-DD` comment
- Include a link to each provider's pricing page in the file header
- When you see prices have changed, update the numbers manually

This is simple, reliable, and what most production tools do.

---

## `load_dotenv()` — how `.env` files work

In [ ]:
from dotenv import load_dotenv
import os

# Reads .env from the current directory (or parent dirs) and loads
# each KEY=VALUE line into os.environ — as if you had exported them in your shell.
load_dotenv()

# Does NOT override variables already set in your shell.
# Shell env vars always win over .env — intentional for production safety.
print(os.getenv("OPENAI_API_KEY", "not set"))

---

## `ModelConfig` — frozen dataclass

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class ModelConfig:
    provider: str            # "openai" | "gemini" | "claude"
    model_id: str            # exact ID sent to the API
    display_name: str        # human-readable label for the UI
    input_cost_per_1k: float
    output_cost_per_1k: float
    supports_vision: bool = field(default=False)

### Design decisions:

**`frozen=True`** — Config is set once at startup, never changed. Immutability prevents accidental mutations.

**`supports_vision: bool`** — Not all models accept image input (e.g. o3-mini is text-only).
Storing this flag means the UI can grey out the image upload button for text-only models.

**`field(default=False)`** — The `field()` call is needed here because `supports_vision` has a
default value but comes after fields without defaults. Without `field()`, Python raises a
`TypeError` about ordering.

**dataclass vs Pydantic here:**
This config is defined *by us, in code*. Values don't come from outside the program.
No validation needed — a frozen dataclass is lighter and more appropriate than Pydantic.

---

## `SUPPORTED_MODELS` — the full catalogue

Instead of one model per provider, we list every text/vision model we want to support.
The UI can offer all of them; users pick what they want to compare.

In [ ]:
SUPPORTED_MODELS: dict[str, ModelConfig] = {
    # key = model_id (also what you send to the API)

    # OpenAI
    "gpt-4o-mini": ModelConfig(
        provider="openai", model_id="gpt-4o-mini", display_name="GPT-4o Mini",
        input_cost_per_1k=0.000150, output_cost_per_1k=0.000600,
        supports_vision=True,
    ),
    "gpt-4o": ModelConfig(
        provider="openai", model_id="gpt-4o", display_name="GPT-4o",
        input_cost_per_1k=0.002500, output_cost_per_1k=0.010000,
        supports_vision=True,
    ),
    # ... etc.
}

DEFAULT_MODELS: dict[str, str] = {
    "openai": "gpt-4o-mini",    # cheapest
    "gemini": "gemini-1.5-flash",  # free tier
    "claude": "claude-haiku-4-5-20251001",  # cheapest
}

### Why key the dict by model_id?

Because lookups happen by model ID: `SUPPORTED_MODELS["gpt-4o-mini"]`.
If we keyed by something else (e.g. display_name), we'd have to search the whole dict every time.

---

## Helper functions

In [ ]:
def get_model(model_id: str) -> ModelConfig:
    """Return config for a model ID. Raises KeyError with a helpful message."""
    if model_id not in SUPPORTED_MODELS:
        available = ", ".join(SUPPORTED_MODELS)
        raise KeyError(f"Unknown model '{model_id}'. Available: {available}")
    return SUPPORTED_MODELS[model_id]


def models_for_provider(provider: str) -> dict[str, ModelConfig]:
    """Return all supported models for a given provider."""
    return {k: v for k, v in SUPPORTED_MODELS.items() if v.provider == provider}

**`models_for_provider`** is a dict comprehension — it filters `SUPPORTED_MODELS` to only entries
where `v.provider == provider`. The UI will call this to populate the model dropdown for each column.

---

## `get_api_key()` — safe key retrieval

In [ ]:
_ENV_VARS: dict[str, str] = {
    "openai": "OPENAI_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "claude": "ANTHROPIC_API_KEY",
}

def get_api_key(provider: str) -> str:
    env_var = _ENV_VARS[provider]
    key = os.getenv(env_var)
    if not key:
        raise EnvironmentError(f"Missing API key: set {env_var} in your .env file.")
    return key

# Fails fast with a clear message instead of a cryptic error deep inside a library.

---

## Pricing cheat sheet (as of 2026-03-30)

| Model | Input / 1M tokens | Output / 1M tokens | Vision? | Free tier? |
|---|---|---|---|---|
| gpt-4o-mini | $0.15 | $0.60 | Yes | No |
| gpt-4o | $2.50 | $10.00 | Yes | No |
| o3-mini | $1.10 | $4.40 | No | No |
| gemini-1.5-flash | $0 | $0 | Yes | Yes (1,500 req/day) |
| gemini-1.5-pro | $1.25 | $5.00 | Yes | No |
| gemini-2.0-flash | $0.10 | $0.40 | Yes | No |
| claude-haiku-4-5 | $0.80 | $4.00 | Yes | No |
| claude-sonnet-4-6 | $3.00 | $15.00 | Yes | No |
| claude-opus-4-6 | $15.00 | $75.00 | Yes | No |

**For this learning project:** Start with `gemini-1.5-flash` (free), `gpt-4o-mini` ($5 in credit lasts forever at this scale), and `claude-haiku-4-5` (same).

---

## Summary

| What | Why |
|---|---|
| Hardcoded pricing with a date + links | No official pricing API exists; this is what production tools do |
| `SUPPORTED_MODELS` keyed by model_id | Fast O(1) lookup; UI can list all available models |
| `supports_vision` field | UI can disable image upload for text-only models |
| `DEFAULT_MODELS` dict | Cheapest/free option per provider; used when user hasn't selected one |
| `get_model()` raises with available models listed | Helpful error for when a new model ID is mistyped |

**Next step:** `utils/timer.py` — a context manager to measure latency, and a cost calculation helper.